# Project 8: Customer Churn Prediction

EDA and modeling workflow for subscription/telco churn. Point `DATA_PATH` at the recommended Kaggle **Telco Customer Churn** CSV when you have it; otherwise the notebook uses the built-in synthetic generator from `src/train.py`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().parent
sys.path.append(str(ROOT / "src"))

from feature_engineering import prepare_features, tenure_monthly_interaction
from train import synthetic_telco_churn

DATA_PATH = None  # e.g. ROOT / "data" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"

if DATA_PATH and Path(DATA_PATH).is_file():
    df = pd.read_csv(DATA_PATH)
else:
    df = synthetic_telco_churn()

df = tenure_monthly_interaction(df)
df.head()

In [ ]:
target = "Churn" if "Churn" in df.columns else "churn"
if df[target].dtype == object:
    churn_rate = (df[target].str.lower().isin(["yes", "true", "1"])).mean()
else:
    churn_rate = df[target].mean()
print(f"Churn rate: {churn_rate:.2%}")
df.describe(include="all").T.head(20)

In [ ]:
num_cols = df.select_dtypes(include="number").columns.tolist()
if target in num_cols:
    num_cols.remove(target)
pair_cols = num_cols[:4] if len(num_cols) >= 4 else num_cols
if pair_cols:
    sns.pairplot(df, vars=pair_cols, hue=target, plot_kws={"alpha": 0.4})
    plt.show()

## Train / explain

- Run `python ../src/train.py --data /path/to/churn.csv` for a full CLI train.
- For SHAP, after training: `import shap` and `shap.summary_plot(shap_values, X_test)` on your holdout matrix.